In [ ]:
import pandas as pd
import numpy as np
import joblib
import pickle
import xgboost as xgb
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings

In [ ]:
# Ignore harmless warnings from statsmodels
warnings.filterwarnings("ignore")

In [ ]:
# ==========================================
# 1. Read and preprocess the test data
# ==========================================
print("Reading test data...")
try:
    df = pd.read_csv('../data/BTCUSD1D-test.csv')
except FileNotFoundError:
    print("Error: File 'btcusd-test.csv' not found.")
    exit()

In [ ]:
# Convert datetime column to datetime format and sort chronologically
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime')
df = df.set_index('datetime')

In [ ]:
# Keep a copy of the full 'close' series for SARIMA before dropping any rows
full_close_series = df['close'].copy()

In [ ]:
# ==========================================
# 2. Feature Engineering for LR and XGBoost
# ==========================================
window_size = 30
print(f"Creating {window_size} lag features for Linear Regression and XGBoost...")

In [ ]:
for i in range(1, window_size + 1):
    df[f'Lag_{i}'] = df['close'].shift(i)

In [ ]:
# Drop the first 30 rows that now contain missing data (NaNs) due to shifting
df_lagged = df.dropna()

In [ ]:
# Extract Features (X) and Actual Prices (y)
feature_cols = [f'Lag_{i}' for i in range(1, window_size + 1)]
X = df_lagged[feature_cols]
actual_prices = df_lagged['close']

In [ ]:
# ==========================================
# 3. Load the pre-trained models
# ==========================================
print("Loading models...")

In [ ]:
# Load Linear Regression
lr_model = joblib.load('models/1_lr_btc_30days_model.pkl')

In [ ]:
# Load SARIMA
with open('models/2_sarima_rolling_btc_model.pkl', 'rb') as f:
    sarima_loaded = pickle.load(f)

In [ ]:
# Load XGBoost
xgb_model = xgb.XGBRegressor()
xgb_model.load_model('models/3_xgboost_btc_30days_model.json')

In [ ]:
# ==========================================
# 4. Generate Predictions (1-step ahead)
# ==========================================
print("Generating predictions based on actual previous prices...")

In [ ]:
# Linear Regression predictions
lr_preds = lr_model.predict(X)

In [ ]:
# XGBoost predictions
xgb_preds = xgb_model.predict(X)

In [ ]:
# SARIMA predictions
# Create a new SARIMAX structure with the full test dataset 
# and apply the learned parameters from the loaded model
sarima_structure = SARIMAX(full_close_series, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0), enforce_stationarity=False, enforce_invertibility=False)
sarima_filtered = sarima_structure.filter(sarima_loaded.params)

In [ ]:
# Predict using dynamic=False to ensure it uses the actual t-1 price to predict t
sarima_full_preds = sarima_filtered.predict(dynamic=False)

In [ ]:
# Align SARIMA predictions with the lagged dataset's dates (dropping the first 30 days)
sarima_preds = sarima_full_preds.loc[actual_prices.index].values

In [ ]:
# ==========================================
# 5. Calculate Average Predicted Price
# ==========================================
avg_preds = (lr_preds + xgb_preds + sarima_preds) / 3

In [ ]:
# ==========================================
# 6. Display Results in Terminal
# ==========================================
# Create a DataFrame to hold all results side-by-side
results_df = pd.DataFrame({
    'Actual Price': actual_prices.values,
    'LR Predict': lr_preds,
    'SARIMA Predict': sarima_preds,
    'XGB Predict': xgb_preds,
    'Average Predict': avg_preds
}, index=actual_prices.index)

In [ ]:
print("\nPrediction Results (1-Step Ahead):")
print(results_df.round(2).head(20)) # Display the first 20 rows

In [ ]:
# ==========================================
# 7. Plot the Results
# ==========================================
print("\nPlotting graph...")
plt.figure(figsize=(15, 8))

In [ ]:
# Plot actual prices with a thick black line
plt.plot(results_df.index, results_df['Actual Price'], label='Actual Price', color='black', linewidth=2)

In [ ]:
# Plot individual models with thinner, dashed/dotted lines
plt.plot(results_df.index, results_df['LR Predict'], label='Linear Predict', color='green', alpha=0.6, linestyle='--')
plt.plot(results_df.index, results_df['SARIMA Predict'], label='SARIMA Predict', color='orange', alpha=0.6, linestyle='-.')
plt.plot(results_df.index, results_df['XGB Predict'], label='XGB Predict', color='blue', alpha=0.6, linestyle=':')

In [ ]:
# Plot the ensemble average with a prominent red line
plt.plot(results_df.index, results_df['Average Predict'], label='Average Predict', color='red', linewidth=2.5)

In [ ]:
plt.title('BTC Price Ensemble Evaluation (LR vs SARIMA vs XGBoost vs Average)')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()